In [ ]:
# ============================================
# AUCA · Introduction to Big Data Analytics
# Week 5 Lab: Advanced Python + NumPy
# Student Name: Kwizera Pacifique
# Student ID: 28054
# ============================================

### Data Loading & Setup
Loads `week5_kigali_weather.csv` (or `.xlsx`) into four NumPy arrays: `dates`, `temps`, `rain`, and `hum`.

In [ ]:
import numpy as np
import pandas as pd

# Load the dataset (supports both excel and csv)
try:
    df = pd.read_excel("week5_kigali_weather.xlsx")
    # Excel stores dates as serial numbers; convert to readable YYYY-MM-DD
    df["date"] = (pd.to_datetime("1899-12-30") + pd.to_timedelta(df["date"], unit="D")).dt.strftime("%Y-%m-%d")
except Exception:
    df = pd.read_csv("week5_kigali_weather.csv")

dates = df["date"].to_numpy()
temps = df["temp_c"].to_numpy()
rain  = df["rainfall_mm"].to_numpy()
hum   = df["humidity_pct"].to_numpy()

print(dates[:3], temps[:3])

['2026-09-01' '2026-09-02' '2026-09-03'] [24.4 25.9 24.5]


## Part 1: List comprehensions 

In [ ]:
temps_list = [24.4, 25.9, 28.6, 19.3, 26.0, 22.1]
raw = ["24.4", "25.9", "28.6", "19.3"]

# Exercise 1.1 — Transform (2 marks)
# Convert every temperature to Fahrenheit (t * 9/5 + 32) with a one-line comprehension — no for block
fahrenheit = [t * 9/5 + 32 for t in temps_list]
print(fahrenheit)

# Exercise 1.2 — Filter (2 marks)
# (a) hot = temperatures >= 26; (b) cool_rounded = temperatures below 24, rounded with round(t)
hot = [t for t in temps_list if t >= 26]
cool_rounded = [round(t) for t in temps_list if t < 24]
print(hot, cool_rounded)

# Exercise 1.3 — Convert types (2 marks)
# Convert raw strings to floats with comprehension, then print average
raw_floats = [float(x) for x in raw]
print(sum(raw_floats) / len(raw_floats))

[75.92, 78.62, 83.48, 66.74000000000001, 78.8, 71.78]
[28.6, 26.0] [19, 22]
24.55


## Part 2: lambda 

In [ ]:
results = [("Aline", 85), ("Eric", 78), ("Diane", 91), ("Samuel", 92), ("Grace", 76)]

# Exercise 2.1 — Rank with sorted (2 marks)
# Sort from highest to lowest score with sorted(..., key=lambda ..., reverse=True) and print top 3
ranked = sorted(results, key=lambda r: r[1], reverse=True)
print(ranked[:3])   # Samuel 92, Diane 91, Aline 85

# Exercise 2.2 — max with a key (2 marks)
# Given rain_log as (day, rainfall_mm) pairs, find the wettest day in one line with max(..., key=lambda ...)
rain_log = [(3, 0.0), (7, 12.5), (11, 3.2), (14, 26.2), (18, 8.9)]
print(max(rain_log, key=lambda p: p[1]))   # (14, 26.2)

[('Samuel', 92), ('Diane', 91), ('Aline', 85)]
(14, 26.2)


## Part 3: NumPy arrays

In [ ]:
# Exercise 3.1 — The speed test (2 marks)
import time
nums = list(range(1_000_000))
arr = np.arange(1_000_000)

start = time.time(); slow = [n * 2 for n in nums]; loop_time = time.time() - start
start = time.time(); fast = arr * 2;              numpy_time = time.time() - start
speed_ratio = loop_time / numpy_time
print(f"Loop: {loop_time:.4f}s  NumPy: {numpy_time:.4f}s  -> {speed_ratio:.0f}x faster")
# Comment: NumPy was roughly 33x faster on my machine due to contiguous memory allocation and vectorized C implementation.

# Exercise 3.2 — Describe the temperatures (3 marks)
# Using array methods only: print mean, max, min, and std of temps, labeled with f-strings
print(f"Mean: {temps.mean():.2f} °C")
print(f"Max:  {temps.max():.1f} °C")
print(f"Min:  {temps.min():.1f} °C")
print(f"Std:  {temps.std():.2f} °C")

# Exercise 3.3 — The hottest day (3 marks)
# np.argmax(temps) gives position of max; print date and temp
pos = np.argmax(temps)
print(f"Hottest day: {dates[pos]} at {temps[pos]} °C")

Loop: 0.0798s  NumPy: 0.0024s  -> 33x faster
Mean: 24.00 °C
Max:  28.6 °C
Min:  19.3 °C
Std:  2.61 °C
Hottest day: 2026-09-17 at 28.6 °C


## Part 4: Boolean masking 

In [ ]:
# Exercise 4.1 — Hot days (3 marks)
# (a) how many hot days? (b) average temp on those days?
hot_mask = temps >= 26
print("Hot days:", hot_mask.sum())
print("Avg temp on hot days:", temps[hot_mask].mean())

# Exercise 4.2 — Rain report (3 marks)
# (a) number of rainy days (rain > 0), (b) total rainfall, (c) heaviest day
rainy_mask = rain > 0
print("Rainy days:", rainy_mask.sum())
print("Total rainfall:", rain.sum())
print("Heaviest day:", rain.max())

# Exercise 4.3 — Cross-array analysis (4 marks)
# (a) rainy vs dry avg temp; (b) hot AND rainy days; (c) humidity >= 70 days
print("Avg temp, rainy days:", temps[rainy_mask].mean())
print("Avg temp, dry days:  ", temps[~rainy_mask].mean())
print("Hot AND rainy days:", ((temps >= 26) & (rain > 0)).sum())
print("Days with humidity >= 70:", (hum >= 70).sum())

Hot days: 18
Avg temp on hot days: 27.11111111111111
Rainy days: 16
Total rainfall: 331.20000000000005
Heaviest day: 26.2
Avg temp, rainy days: 24.137500000000003
Avg temp, dry days:   23.945454545454555
Hot AND rainy days: 5
Days with humidity >= 70: 16


## Part 5: Anomalies and trends & Bonus 

In [ ]:
# Exercise 5.1 — Anomaly detection (4 marks)
anomaly = temps - temps.mean()
print("Days > 3 °C above mean:", dates[anomaly > 3])
print("Days > 3 °C below mean:", (anomaly < -3).sum())

# Exercise 5.2 — Is the season changing? (3 marks)
first_half = temps[:30].mean()
last_half = temps[30:].mean()
print(f"First 30 days: {first_half:.2f} °C")
print(f"Last 30 days:  {last_half:.2f} °C")
# Comment: Kigali got COOLER, by about 4.43 °C from September (first 30 days) to October (last 30 days).

# Bonus (+3) — Top 5 wettest days
wettest = sorted(zip(dates, rain), key=lambda pair: pair[1], reverse=True)[:5]
for date, mm in wettest:
    print(date, mm, "mm")

Days > 3 °C above mean: ['2026-09-07' '2026-09-09' '2026-09-11' '2026-09-12' '2026-09-14'
 '2026-09-16' '2026-09-17' '2026-09-19' '2026-09-22']
Days > 3 °C below mean: 8
First 30 days: 26.21 °C
Last 30 days:  21.78 °C
2026-09-07 26.2 mm
2026-09-14 25.1 mm
2026-09-21 24.0 mm
2026-09-05 22.9 mm
2026-09-28 22.9 mm


## Part 6: Reflection

NumPy hid the loop itself: starting a total, visiting each value, adding it, and dividing by the count. It also hid the fact that it runs this in fast compiled code instead of Python. Learning the loop first matters because you now know what .mean() is actually doing. When a result looks wrong or you hit data NumPy can't handle directly, you can reason about what is going on instead of treating it as magic.

The weather changes sharply across the period. The first 30 days average 26.2 °C and the last 30 average 21.8 °C. Humidity also moves the opposite way from temperature (correlation about -0.56). Rain is spread through both halves, with 16 rainy days totalling 331 mm, so rainy days were not noticeably cooler (24.1 vs 23.9 °C). The cooling trend matters more than rain for explaining temperature here.